In [10]:
import subprocess
import os
import csv
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from pathlib import Path

In [11]:
os.chdir("C:/Users/ed/Development/dhzw/8-sensitivity-analysis")

In [12]:
# function to print elapsed time
def calculate_elapsed_time(start_time, end_time):
    elapsed_time = end_time - start_time
    total_seconds = elapsed_time.total_seconds()
    seconds = int(total_seconds % 60)
    total_minutes = total_seconds // 60
    minutes = int(total_minutes % 60)
    hours = int(total_minutes // 60)
    
    seconds = str(seconds).zfill(2)
    minutes = str(minutes).zfill(2)
    hours = str(hours).zfill(2)

    return str(hours) + ':' + str(minutes) + ':' + str(seconds)

# Initialise

### Read parameter sets to simulate

In [13]:
# read parameters to run
table = pd.read_csv('data/parametersets_to_run.csv')
table

,alphaWalk,alphaBike,alphaCarDriver,alphaCarPassenger,alphaBus,alphaTrain,betaTimeWalk,betaTimeBike,betaTimeCarDriver,betaTimeCarPassenger,...,betaCostCarPassenger,betaCostBus,betaCostTrain,betaTimeWalkTransport,betaChangesTransport,used_parameter_label,used_parameter_value,carCostKm,ptCostKm,ptBaseCost
0,-10.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-10.0,0.25,0.187,1.08
1,-8.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-8.0,0.25,0.187,1.08
2,-6.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-6.0,0.25,0.187,1.08
3,-4.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-4.0,0.25,0.187,1.08
4,-2.0,0.0,0.0,0.0,0.0,0.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaWalk,-2.0,0.25,0.187,1.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,0.0,0.0,0.0,0.0,0.0,2.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,2.0,0.25,0.187,1.08
62,0.0,0.0,0.0,0.0,0.0,4.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,4.0,0.25,0.187,1.08
63,0.0,0.0,0.0,0.0,0.0,6.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,6.0,0.25,0.187,1.08
64,0.0,0.0,0.0,0.0,0.0,8.0,-0.04,-0.03,-0.02,-0.02,...,-0.15,-0.1,-0.1,-0.03,-0.3,alphaTrain,8.0,0.25,0.187,1.08


### Initialise empty the output dataframe

In [14]:
# Initialise dataframe for mode choice distributions (1 line = 1 distribution = 1 iteration)
df = pd.DataFrame(columns=["CAR_DRIVER", "CAR_PASSENGER", "BIKE", "BUS_TRAM", "TRAIN", "WALK"])

for i in range(0, len(table)):
    df.to_csv("output/iterations/output_proportions_" + str(i) + ".csv", index=False)

In [15]:

def intialise_java():

    jdk_11_path = r"C:\Program Files\Java\jdk-11"

    os.environ["JAVA_HOME"] = jdk_11_path
    bin_path = os.path.join(jdk_11_path, "bin")
    os.environ["PATH"] = bin_path + os.pathsep + os.environ.get("PATH", "")

    # 4. Verification
    try:
        result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
        print("Java Version Output:\n", result.stderr) 
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")
    except FileNotFoundError:
        print("Java executable not found in the updated PATH.")

intialise_java()


Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



# Call Sim2APL iteratively

### Prepare the Java command

In [16]:
# baseline of the Java command
config = "--config src/main/resources/config_DHZW_stt_senstivity.toml"


output = "-o output/deskrun"
parametersPath = "--parameter_file ../8-sensitivity-analysis/data/parametersets_to_run.csv"

base_cmd = "java -cp target/sim2apl-dhzw-simulation-1.0-SNAPSHOT-jar-with-dependencies.jar main.java.nl.uu.iss.ga.Simulation" + " " + config +  " " + parametersPath + " --use_random_seed true"


    
# set current directory the folder of Sim2APL so I can execute the jar with the correct paths
if(os.path.basename(os.getcwd()) != 'DHZW-simulation_Sim-2APL'):
    # new_directory = os.path.join(os.getcwd(), '7-simulation-Sim-2APL')
    # new_directory = new_directory.replace('\\', '/')
    os.chdir("C:/Users/ed/Development/dhzw/7-simulation-Sim-2APL") #sorry folks, I'm outta time!

### Main loop to call the simulations

In [17]:
idx_restart = 0 # in case somethings breaks it is easy to continue

In [18]:
time_beginning_all = datetime.now()

for idx, row in tqdm(table.iterrows(), total=table.shape[0]):
    if idx >= idx_restart:
        # repeat each parameter set 5 times
        for iteration in range(0, 5):

            # output file for this iteration
            outputPath = f'--output_file=../8-sensitivity-analysis/output/iterations/output_proportions_{idx}.csv'
            cmd = base_cmd + ' ' + outputPath
            
            # parameterset to use
            arg = f'--parameterset_index={idx}'
            full_command = f'{cmd} {arg}'
            
            time_beginning_iteration = datetime.now()
            
            print('parameter number: ' + str(idx) + ' - iteration: ' + str(iteration) + ' START || Time now: ' + time_beginning_iteration.strftime("%H:%M:%S"))
                                    
            # Capture the output of the Java program
            try:
                output = subprocess.check_output(full_command, stderr=subprocess.STDOUT, universal_newlines=True)
            except subprocess.CalledProcessError as e:
                print(f"Java program exited with non-zero return code: {e.returncode}")
                print(f"Error message: {e.output}")
                exit(1)
                
            time_end_iteration = datetime.now()
         
            print('parameter number: ' + str(idx) + ' - iteration: ' + str(iteration) + ' END   || Time now: ' + time_end_iteration.strftime("%H:%M:%S") + ' | Duration iteration: ' + calculate_elapsed_time(time_beginning_iteration, time_end_iteration) + ' | Elapsed from start: ' + calculate_elapsed_time(time_beginning_all, time_end_iteration))

  0%|          | 0/66 [00:00<?, ?it/s]

parameter number: 0 - iteration: 0 START || Time now: 13:01:41
parameter number: 0 - iteration: 0 END   || Time now: 13:01:58 | Duration iteration: 00:00:16 | Elapsed from start: 00:00:16
parameter number: 0 - iteration: 1 START || Time now: 13:01:58
parameter number: 0 - iteration: 1 END   || Time now: 13:02:15 | Duration iteration: 00:00:16 | Elapsed from start: 00:00:33
parameter number: 0 - iteration: 2 START || Time now: 13:02:15
parameter number: 0 - iteration: 2 END   || Time now: 13:02:34 | Duration iteration: 00:00:18 | Elapsed from start: 00:00:52
parameter number: 0 - iteration: 3 START || Time now: 13:02:34
parameter number: 0 - iteration: 3 END   || Time now: 13:02:51 | Duration iteration: 00:00:17 | Elapsed from start: 00:01:10
parameter number: 0 - iteration: 4 START || Time now: 13:02:51


  2%|▏         | 1/66 [01:28<1:35:27, 88.12s/it]

parameter number: 0 - iteration: 4 END   || Time now: 13:03:09 | Duration iteration: 00:00:18 | Elapsed from start: 00:01:28
parameter number: 1 - iteration: 0 START || Time now: 13:03:09
parameter number: 1 - iteration: 0 END   || Time now: 13:03:27 | Duration iteration: 00:00:17 | Elapsed from start: 00:01:45
parameter number: 1 - iteration: 1 START || Time now: 13:03:27
parameter number: 1 - iteration: 1 END   || Time now: 13:03:46 | Duration iteration: 00:00:18 | Elapsed from start: 00:02:04
parameter number: 1 - iteration: 2 START || Time now: 13:03:46
parameter number: 1 - iteration: 2 END   || Time now: 13:04:04 | Duration iteration: 00:00:18 | Elapsed from start: 00:02:22
parameter number: 1 - iteration: 3 START || Time now: 13:04:04
parameter number: 1 - iteration: 3 END   || Time now: 13:04:22 | Duration iteration: 00:00:17 | Elapsed from start: 00:02:40
parameter number: 1 - iteration: 4 START || Time now: 13:04:22


  3%|▎         | 2/66 [02:59<1:36:02, 90.04s/it]

...

parameter number: 63 - iteration: 4 END   || Time now: 14:30:51 | Duration iteration: 00:00:18 | Elapsed from start: 01:29:09
parameter number: 64 - iteration: 0 START || Time now: 14:30:51
parameter number: 64 - iteration: 0 END   || Time now: 14:31:08 | Duration iteration: 00:00:16 | Elapsed from start: 01:29:26
parameter number: 64 - iteration: 1 START || Time now: 14:31:08
parameter number: 64 - iteration: 1 END   || Time now: 14:31:25 | Duration iteration: 00:00:17 | Elapsed from start: 01:29:44
parameter number: 64 - iteration: 2 START || Time now: 14:31:25
parameter number: 64 - iteration: 2 END   || Time now: 14:31:44 | Duration iteration: 00:00:18 | Elapsed from start: 01:30:02
parameter number: 64 - iteration: 3 START || Time now: 14:31:44
parameter number: 64 - iteration: 3 END   || Time now: 14:32:02 | Duration iteration: 00:00:17 | Elapsed from start: 01:30:20
parameter number: 64 - iteration: 4 START || Time now: 14:32:02


 98%|█████████▊| 65/66 [1:30:38<01:29, 89.56s/it]

parameter number: 64 - iteration: 4 END   || Time now: 14:32:20 | Duration iteration: 00:00:18 | Elapsed from start: 01:30:38
parameter number: 65 - iteration: 0 START || Time now: 14:32:20
parameter number: 65 - iteration: 0 END   || Time now: 14:32:38 | Duration iteration: 00:00:17 | Elapsed from start: 01:30:56
parameter number: 65 - iteration: 1 START || Time now: 14:32:38
parameter number: 65 - iteration: 1 END   || Time now: 14:32:56 | Duration iteration: 00:00:18 | Elapsed from start: 01:31:14
parameter number: 65 - iteration: 2 START || Time now: 14:32:56
parameter number: 65 - iteration: 2 END   || Time now: 14:33:14 | Duration iteration: 00:00:18 | Elapsed from start: 01:31:32
parameter number: 65 - iteration: 3 START || Time now: 14:33:14
parameter number: 65 - iteration: 3 END   || Time now: 14:33:31 | Duration iteration: 00:00:17 | Elapsed from start: 01:31:49
parameter number: 65 - iteration: 4 START || Time now: 14:33:31


100%|██████████| 66/66 [1:32:07<00:00, 83.75s/it]

parameter number: 65 - iteration: 4 END   || Time now: 14:33:49 | Duration iteration: 00:00:17 | Elapsed from start: 01:32:07
